## Leave-One-Device-Out（LODO，对齐跨协议 LOPO）

7 路融合、`y=final_type`、`window=128`、`eval_step=2`（与官方 UltraLite 一致）。

1. **训练**：清零留出设备 \(d\) 的特征（模型从未见过该设备通道）
2. **验证 / 早停**：Full-7
3. **测试**：Full-7（\(d\) 在测试时首次带着真实特征出现）
4. **主表**：Target Macro-F1（7 折 mean±std）、Worst、Gap = Source − Target

**权重缓存**：每折每模型保存到 `mmm/{device}/{model}.pth`（含 `meta.json`、`scaler.npz`），下次自动跳过已训折。

结果：`lodo_leave_one_device_out.csv`（`python _run_lodo.py` 生成）。


In [ ]:
# ===== LODO：读 Leave-One-Device-Out 结果（mean±std 四项指标） =====
import os
import pandas as pd

_ROOT = os.path.abspath(os.getcwd())
for _cand in [_ROOT, os.path.dirname(_ROOT), r'E:\apt\YES\2']:
    if os.path.isfile(os.path.join(_cand, 'lodo_leave_one_device_out.csv')):
        _ROOT = _cand
        os.chdir(_ROOT)
        break

csv_p = os.path.join(_ROOT, 'lodo_leave_one_device_out.csv')
metrics_p = os.path.join(_ROOT, 'lodo_leave_one_device_metrics.csv')
assert os.path.isfile(csv_p), (
    f'缺少 {csv_p}\n请先运行: python _run_lodo.py  （约 2–3 小时，7 设备 × 8 模型）')

df_main = pd.read_csv(csv_p, index_col=0)
proto_cols = [c for c in df_main.columns if c.startswith('T_')]
METRICS = ['Accuracy', 'Precision', 'Recall', 'F1']


def fmt_pm(mean, std, digits=4):
    return f'{mean:.{digits}f}±{std:.{digits}f}'


dm = pd.read_csv(metrics_p, index_col=0).sort_values('F1_mean', ascending=False)

print('LODO Target metrics (7-fold mean±std; train w/o device-d → test full-7)\n')
hdr = f'{"Model":18s}  ' + '  '.join(f'{m:>16s}' for m in METRICS)
print(hdr)
print('-' * len(hdr))
for model, row in dm.iterrows():
    bits = [fmt_pm(row[f'{m}_mean'], row[f'{m}_std']) for m in METRICS]
    print(f'{model:18s}  ' + '  '.join(f'{b:>16s}' for b in bits))

print('\n' + '=' * 72)
print('Per-model detail')
print('=' * 72)
for model, row in dm.iterrows():
    print(f'\n{model}')
    for m in METRICS:
        print(f'  {m:10s}: {fmt_pm(row[f"{m}_mean"], row[f"{m}_std"])}')

print('\nExcel paste (tab-separated)')
print('Model\t' + '\t'.join(METRICS))
for model, row in dm.iterrows():
    print(model + '\t' + '\t'.join(fmt_pm(row[f'{m}_mean'], row[f'{m}_std']) for m in METRICS))

cols = [c for c in ['Source_F1', 'Target_mean', 'Target_std', 'Target_worst', 'Gap_mean', 'Gap_worst']
        if c in df_main.columns]
show = df_main.sort_values('Target_mean', ascending=False)
print('\nLODO F1 summary')
print(show[cols].round(4).to_string())
print('\nPer-device Target F1:')
print(show[proto_cols].round(4).to_string())


## Leave-k-Devices-Out（k=2 / 3 / 4）

在 LODO（留 1 设备）基础上的扩展（**均在 E 盘 `E:\apt\YES\2` 运行**）：

| k | 组合数 | 训练 | 测试 |
|---|--------|------|------|
| 2 | 21 | 清零 2 路设备 | Full-7 |
| 3 | 35 | 清零 3 路设备 | Full-7 |
| 4 | 35 | 清零 4 路设备 | Full-7 |

运行：`Set-Location E:\apt\YES\2; python _run_lodo_k.py`

缓存：`E:\apt\YES\2\_tmp\lodo_cache\lodo_k_cache\`

结果：`lodo_leave{k}_device_metrics.csv`、`lodo_leave_k_device_metrics.csv`

In [ ]:
# ===== Leave-k-Devices-Out：k=2/3/4 四项指标 mean±std =====
# 运行本单元即可在下方看到表格（需先跑完 _run_lodo_k.py）
import os
import pandas as pd
from IPython.display import display, Markdown

_ROOT = r'E:\apt\YES\2'
os.chdir(_ROOT)

METRICS = ['Accuracy', 'Precision', 'Recall', 'F1']
K_VALUES = [2, 3, 4]


def fmt_pm(mean, std, digits=4):
    return f'{mean:.{digits}f}±{std:.{digits}f}'


def show_k_table(k):
    metrics_p = os.path.join(_ROOT, f'lodo_leave{k}_device_metrics.csv')
    display(Markdown(f'### Leave-{k}-Devices-Out（train 清零 {k} 设备 → test Full-7）'))
    if not os.path.isfile(metrics_p):
        display(Markdown(f'**尚未生成** `{metrics_p}` — k={k} 仍在跑或未完成。日志：`_lodo_k34_log.txt`'))
        cache_dir = os.path.join(_ROOT, '_tmp', 'lodo_cache', 'lodo_k_cache')
        if os.path.isdir(cache_dir):
            n = len([f for f in os.listdir(cache_dir) if f.startswith(f'k{k}_') and f.endswith('.npz')])
            display(Markdown(f'缓存进度：已完成 **{n}** / {35 if k >= 3 else 21} 组合'))
        return
    dm = pd.read_csv(metrics_p, index_col=0).sort_values('F1_mean', ascending=False)
    n_folds = int(dm['n_folds'].iloc[0]) if 'n_folds' in dm.columns else len(dm)
    rows = []
    for model, row in dm.iterrows():
        rows.append({
            'Model': model,
            **{m: fmt_pm(row[f'{m}_mean'], row[f'{m}_std']) for m in METRICS},
        })
    tbl = pd.DataFrame(rows).set_index('Model')
    display(Markdown(f'**{n_folds}-combo mean±std**'))
    display(tbl)
    display(Markdown('**Excel 粘贴（Tab 分隔）**'))
    print('Model\t' + '\t'.join(METRICS))
    for _, r in tbl.reset_index().iterrows():
        print(f'{r["Model"]}\t' + '\t'.join(r[m] for m in METRICS))


for k in K_VALUES:
    show_k_table(k)

combined_p = os.path.join(_ROOT, 'lodo_leave_k_device_metrics.csv')
if os.path.isfile(combined_p):
    display(Markdown('### 合并表 lodo_leave_k_device_metrics.csv'))
    display(pd.read_csv(combined_p)[['k', 'model', 'F1_mean', 'F1_std', 'n_folds']].round(4))